### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="wine_quality",
    dataset_year="2009",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C56S3T",
    download_description="""
Download from UCI and uzip data to a predefined folder.
mkdir -p local-data-warehouse/wine_quality/ && wget -P local-data-warehouse/wine_quality/ https://archive.ics.uci.edu/static/public/186/wine+quality.zip && unzip local-data-warehouse/wine_quality/wine+quality.zip -d local-data-warehouse/wine_quality/ && rm local-data-warehouse/wine_quality/wine+quality.zip
""",
    # References
    academic_reference_bibtex=r"""@article{cortez2009modeling,
  title={Modeling wine preferences by data mining from physicochemical properties},
  author={Cortez, Paulo and Cerdeira, Ant{\'o}nio and Almeida, Fernando and Matos, Telmo and Reis, Jos{\'e}},
  journal={Decision support systems},
  volume={47},
  number={4},
  pages={547--553},
  year={2009},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="cortez2009modeling",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We combine the original datasets for red and white wine into a single dataset with an additional column indicating the type of wine (red or white).
- We treat the task as a regression problem, following the original work and because the target is the median wine quality (of at least 3 evaluations by experts).
- Anomaly: the data has a high number of duplicates (18%).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="median_wine_quality",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd

df_red = pd.read_csv(dataset_mold.path / "winequality-red.csv", sep=";")
df_white = pd.read_csv(dataset_mold.path / "winequality-white.csv", sep=";")
df_red["wine_color"] = "red"
df_white["wine_color"] = "white"
df = pd.concat([df_white, df_red], ignore_index=True) 
df.columns = df.columns.str.replace(" ", "_")
df["wine_color"] = df["wine_color"].astype("category")

target_feature = "median_wine_quality"
df = df.rename(columns={"quality": target_feature})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()